# Seasonal Agriculture Performance Analysis
This notebook conducts a comprehensive seasonal performance analysis covering data cleaning, exploratory data analysis, resource optimization, irrigation/crop dynamics, statistical testing, predictive machine learning modeling, and key recommendations.

In [4]:
# =====================================================================
# STEP 1: IMPORTING REQUIRED LIBRARIES AND CONFIGURATIONS
# =====================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr, spearmanr
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)
print("Libraries imported successfully!")

Libraries imported successfully!


In [5]:
# =====================================================================
# STEP 2: LOADING THE DATASET
# =====================================================================
file_path = "/content/seasonal_agriculture_performance_dataset.csv"
df = pd.read_csv(file_path)
print("Dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
display(df.head())

Dataset loaded successfully!
Rows: 4000
Columns: 28


,Farm_ID,State,District,Crop,Season,Farm_Area_Hectares,Rainfall_mm,Avg_Temperature_C,Humidity_pct,Sunlight_Hours_Day,Soil_pH,Soil_Moisture_pct,Nitrogen_kg_ha,Phosphorus_kg_ha,Potassium_kg_ha,Irrigation_Method,Fertilizer_kg_ha,Pesticide_Litre_ha,Seed_Quality_Score,Yield_Tonnes_Ha,Production_Tonnes,Market_Price_INR_Tonne,Total_Cost_INR,Revenue_INR,Profit_INR,Water_Used_m3,Water_Efficiency_t_per_1000m3,Disease_Pest_Risk_pct
0,SF10001,Andhra Pradesh,Rajkot,Wheat,Kharif,0.53,486.40,24.30,69.10,5.80,7.14,32.80,75.80,61.30,91.00,Drip,242.70,4.33,0.76,2.46,1.30,23700,42662,30810,-11852,237,5.49,55.30
1,SF10002,Maharashtra,Nalgonda,Maize,Kharif,6.53,855.40,25.70,78.40,5.10,5.20,32.00,82.10,35.40,121.80,Flood,271.40,6.86,0.94,0.30,1.96,20613,492351,40401,-451950,4953,0.40,49.90
2,SF10003,Telangana,Warangal,Pulses,Rabi,4.86,455.60,23.90,56.40,10.30,5.82,8.10,118.50,51.50,117.90,Drip,162.60,6.08,0.98,0.52,2.53,75283,315210,190466,-124744,1275,1.98,33.70
3,SF10004,Telangana,Indore,Rice,Kharif,4.50,753.20,29.70,65.20,6.30,5.98,23.70,138.60,55.60,115.50,Rainfed,182.20,6.39,0.74,1.84,8.28,24244,296444,200740,-95704,3834,2.16,50.60
4,SF10005,Karnataka,Ludhiana,Maize,Zaid,4.21,101.60,34.10,55.20,6.50,6.90,29.30,124.10,69.00,121.40,Flood,244.90,7.61,0.90,2.21,9.30,20818,337959,193607,-144352,3287,2.83,32.30


In [6]:
# =====================================================================
# STEP 3: DATASET OVERVIEW AND INFORMATION
# =====================================================================
print("Dataset Shape:")
print(df.shape)
print("\nColumn Names:")
print(df.columns.tolist())
print("\nDataset Information:")
df.info()

Dataset Shape:
(4000, 28)

Column Names:
['Farm_ID', 'State', 'District', 'Crop', 'Season', 'Farm_Area_Hectares', 'Rainfall_mm', 'Avg_Temperature_C', 'Humidity_pct', 'Sunlight_Hours_Day', 'Soil_pH', 'Soil_Moisture_pct', 'Nitrogen_kg_ha', 'Phosphorus_kg_ha', 'Potassium_kg_ha', 'Irrigation_Method', 'Fertilizer_kg_ha', 'Pesticide_Litre_ha', 'Seed_Quality_Score', 'Yield_Tonnes_Ha', 'Production_Tonnes', 'Market_Price_INR_Tonne', 'Total_Cost_INR', 'Revenue_INR', 'Profit_INR', 'Water_Used_m3', 'Water_Efficiency_t_per_1000m3', 'Disease_Pest_Risk_pct']

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 28 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Farm_ID                        4000 non-null   object 
 1   State                          4000 non-null   object 
 2   District                       4000 non-null   object 
 3   Crop            

In [ ]:
# =====================================================================
# STEP 4: STATISTICAL SUMMARY OF NUMERICAL VARIABLES
# =====================================================================
display(df.describe().T)

In [ ]:
# =====================================================================
# STEP 5: DETECTING AND SUMMARIZING MISSING VALUES
# =====================================================================
missing_values = df.isnull().sum()
missing_table = pd.DataFrame({
    'Column': missing_values.index,
    'Missing Values': missing_values.values,
    'Missing %': (missing_values.values / len(df)) * 100
}).sort_values(by='Missing Values', ascending=False)
display(missing_table)

In [ ]:
# =====================================================================
# STEP 6: IDENTIFYING AND CLEANING DUPLICATE RECORDS
# =====================================================================
duplicates = df.duplicated().sum()
print("Number of duplicate rows:", duplicates)
if duplicates > 0:
    df = df.drop_duplicates()
    print("Dataset shape after removing duplicates:", df.shape)

In [ ]:
# =====================================================================
# STEP 7: CHECKING UNIQUE CATEGORICAL CARDINALITIES
# =====================================================================
categorical_columns = df.select_dtypes(include='object').columns
for col in categorical_columns:
    print(f"\nUnique values for column '{col}':")
    print(df[col].unique())

In [ ]:
# =====================================================================
# STEP 8: DISTRIBUTION ANALYSIS OF SEASONS
# =====================================================================
print("Seasons available:")
print(df['Season'].value_counts())

plt.figure(figsize=(8,5))
sns.countplot(data=df, x='Season')
plt.title("Number of Farms by Season")
plt.xlabel("Season")
plt.ylabel("Number of Farms")
plt.show()

In [ ]:
# =====================================================================
# STEP 9: CROP PROFILE AND FREQUENCY DISTRIBUTION
# =====================================================================
print("Number of different crops:", df['Crop'].nunique())
print("\nCrop distribution:")
print(df['Crop'].value_counts())

plt.figure(figsize=(10,5))
sns.countplot(data=df, x='Crop', order=df['Crop'].value_counts().index)
plt.title("Crop Distribution")
plt.xlabel("Crop")
plt.ylabel("Number of Farms")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# =====================================================================
# STEP 10: STATE-LEVEL AGRICULTURAL DISTRIBUTION
# =====================================================================
plt.figure(figsize=(12,6))
sns.countplot(data=df, y='State', order=df['State'].value_counts().index)
plt.title("Agricultural Records by State")
plt.xlabel("Number of Farms")
plt.ylabel("State")
plt.show()

In [ ]:
# =====================================================================
# STEP 11: COMPREHENSIVE SEASONAL PERFORMANCE METRICS
# =====================================================================
seasonal_summary = df.groupby('Season').agg({
    'Farm_Area_Hectares': 'mean',
    'Rainfall_mm': 'mean',
    'Avg_Temperature_C': 'mean',
    'Humidity_pct': 'mean',
    'Sunlight_Hours_Day': 'mean',
    'Soil_Moisture_pct': 'mean',
    'Fertilizer_kg_ha': 'mean',
    'Pesticide_Litre_ha': 'mean',
    'Yield_Tonnes_Ha': 'mean',
    'Production_Tonnes': 'mean',
    'Total_Cost_INR': 'mean',
    'Revenue_INR': 'mean',
    'Profit_INR': 'mean',
    'Water_Used_m3': 'mean',
    'Water_Efficiency_t_per_1000m3': 'mean',
    'Disease_Pest_Risk_pct': 'mean'
}).round(2)
display(seasonal_summary)

In [ ]:
# =====================================================================
# STEP 12: CROP YIELD COMPARISON BY SEASON
# =====================================================================
plt.figure(figsize=(8,5))
sns.barplot(data=df, x='Season', y='Yield_Tonnes_Ha', estimator='mean')
plt.title("Average Crop Yield by Season")
plt.xlabel("Season")
plt.ylabel("Average Yield (Tonnes/Ha)")
plt.show()

In [ ]:
# =====================================================================
# STEP 13: AVERAGE PROFITABILITY BY SEASON
# =====================================================================
plt.figure(figsize=(8,5))
sns.barplot(data=df, x='Season', y='Profit_INR', estimator='mean')
plt.title("Average Profit by Season")
plt.xlabel("Season")
plt.ylabel("Average Profit (INR)")
plt.show()

In [ ]:
# =====================================================================
# STEP 14: REVENUE POTENTIAL COMPARED ACROSS SEASONS
# =====================================================================
plt.figure(figsize=(8,5))
sns.barplot(data=df, x='Season', y='Revenue_INR', estimator='mean')
plt.title("Average Revenue by Season")
plt.xlabel("Season")
plt.ylabel("Average Revenue (INR)")
plt.show()

In [ ]:
# =====================================================================
# STEP 15: TOTAL COST INCURRED BY SEASON
# =====================================================================
plt.figure(figsize=(8,5))
sns.barplot(data=df, x='Season', y='Total_Cost_INR', estimator='mean')
plt.title("Average Agricultural Cost by Season")
plt.xlabel("Season")
plt.ylabel("Average Cost (INR)")
plt.show()

In [ ]:
# =====================================================================
# STEP 16: AVERAGE CROP PRODUCTION (TONNES) BY SEASON
# =====================================================================
plt.figure(figsize=(8,5))
sns.barplot(data=df, x='Season', y='Production_Tonnes', estimator='mean')
plt.title("Average Production by Season")
plt.xlabel("Season")
plt.ylabel("Production (Tonnes)")
plt.show()

In [ ]:
# =====================================================================
# STEP 17: SEASONAL ENVIRONMENTAL CONDITION VARIATIONS
# =====================================================================
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x='Season', y='Rainfall_mm')
plt.title("Rainfall Distribution Across Seasons")
plt.xlabel("Season")
plt.ylabel("Rainfall (mm)")
plt.show()

plt.figure(figsize=(8,5))
sns.boxplot(data=df, x='Season', y='Avg_Temperature_C')
plt.title("Temperature Distribution Across Seasons")
plt.xlabel("Season")
plt.ylabel("Average Temperature (°C)")
plt.show()

plt.figure(figsize=(8,5))
sns.boxplot(data=df, x='Season', y='Humidity_pct')
plt.title("Humidity Distribution Across Seasons")
plt.xlabel("Season")
plt.ylabel("Humidity (%)")
plt.show()

In [ ]:
# =====================================================================
# STEP 18: SOIL COMPOSITION AND PARAMETERS BY SEASON
# =====================================================================
soil_variables = ['Soil_pH', 'Soil_Moisture_pct', 'Nitrogen_kg_ha', 'Phosphorus_kg_ha', 'Potassium_kg_ha']
for col in soil_variables:
    plt.figure(figsize=(8,5))
    sns.boxplot(data=df, x='Season', y=col)
    plt.title(f"{col} Across Seasons")
    plt.xlabel("Season")
    plt.ylabel(col)
    plt.show()

In [ ]:
# =====================================================================
# STEP 19: FARM INPUT RESOURCE UTILIZATION BY SEASON
# =====================================================================
resource_variables = ['Fertilizer_kg_ha', 'Pesticide_Litre_ha', 'Water_Used_m3']
for col in resource_variables:
    plt.figure(figsize=(8,5))
    sns.barplot(data=df, x='Season', y=col, estimator='mean')
    plt.title(f"Average {col} by Season")
    plt.xlabel("Season")
    plt.ylabel(col)
    plt.show()

In [ ]:
# =====================================================================
# STEP 20: IRRIGATION PATTERN DISTRIBUTIONS OVER SEASONS
# =====================================================================
irrigation_season = pd.crosstab(df['Season'], df['Irrigation_Method'])
print(irrigation_season)

plt.figure(figsize=(10,6))
sns.heatmap(irrigation_season, annot=True, fmt='d', cmap='YlGnBu')
plt.title("Irrigation Methods Across Seasons")
plt.show()

In [ ]:
# =====================================================================
# STEP 21: IMPACT OF IRRIGATION METHOD ON YIELD
# =====================================================================
irrigation_yield = df.groupby('Irrigation_Method')['Yield_Tonnes_Ha'].mean().sort_values(ascending=False)
print(irrigation_yield)

plt.figure(figsize=(9,5))
sns.barplot(x=irrigation_yield.values, y=irrigation_yield.index)
plt.title("Average Yield by Irrigation Method")
plt.xlabel("Yield (Tonnes/Ha)")
plt.ylabel("Irrigation Method")
plt.show()

In [ ]:
# =====================================================================
# STEP 22: FINANCIAL RETURNS BY IRRIGATION APPROACHES
# =====================================================================
plt.figure(figsize=(9,5))
sns.boxplot(data=df, x='Irrigation_Method', y='Profit_INR')
plt.title("Profit Distribution by Irrigation Method")
plt.xlabel("Irrigation Method")
plt.ylabel("Profit (INR)")
plt.xticks(rotation=30)
plt.show()

In [ ]:
# =====================================================================
# STEP 23: CROP PERFORMANCE HEATMAP BY SEASON
# =====================================================================
crop_season_yield = df.pivot_table(values='Yield_Tonnes_Ha', index='Crop', columns='Season', aggfunc='mean')
display(crop_season_yield)

plt.figure(figsize=(12,7))
sns.heatmap(crop_season_yield, annot=True, fmt='.2f', cmap='YlGnBu')
plt.title("Average Yield by Crop and Season")
plt.show()

In [ ]:
# =====================================================================
# STEP 24: NET CROP MARGIN AND PROFIT PROFILE
# =====================================================================
crop_profit = df.groupby('Crop')['Profit_INR'].mean().sort_values(ascending=False)
print(crop_profit)

plt.figure(figsize=(10,6))
sns.barplot(x=crop_profit.values, y=crop_profit.index)
plt.title("Average Profit by Crop")
plt.xlabel("Average Profit (INR)")
plt.ylabel("Crop")
plt.show()

In [ ]:
# =====================================================================
# STEP 25: INTERACTIVE PROFITABILITY MATRIX (CROP x SEASON)
# =====================================================================
season_crop_profit = df.pivot_table(values='Profit_INR', index='Crop', columns='Season', aggfunc='mean')
display(season_crop_profit)

plt.figure(figsize=(12,7))
sns.heatmap(season_crop_profit, annot=True, fmt='.0f', cmap='RdYlGn', center=0)
plt.title("Average Profit by Crop and Season")
plt.show()

In [ ]:
# =====================================================================
# STEP 26: IDENTIFYING BEST PERFORMING SEASONS
# =====================================================================
best_season_yield = seasonal_summary['Yield_Tonnes_Ha'].idxmax()
best_season_profit = seasonal_summary['Profit_INR'].idxmax()
best_season_revenue = seasonal_summary['Revenue_INR'].idxmax()
best_season_water = seasonal_summary['Water_Efficiency_t_per_1000m3'].idxmax()

print("Best season for average yield:", best_season_yield)
print("Best season for average profit:", best_season_profit)
print("Best season for average revenue:", best_season_revenue)
print("Best season for water efficiency:", best_season_water)

In [ ]:
# =====================================================================
# STEP 27: IDENTIFYING LOWEST PERFORMING SEASONS
# =====================================================================
worst_season_yield = seasonal_summary['Yield_Tonnes_Ha'].idxmin()
worst_season_profit = seasonal_summary['Profit_INR'].idxmin()

print("Lowest average yield season:", worst_season_yield)
print("Lowest average profit season:", worst_season_profit)

In [ ]:
# =====================================================================
# STEP 28: NET PROFITABILITY STATUS BINARY CLASSIFICATION
# =====================================================================
df['Profit_Status'] = np.where(df['Profit_INR'] >= 0, 'Profitable', 'Loss')
print(df['Profit_Status'].value_counts())

plt.figure(figsize=(7,5))
sns.countplot(data=df, x='Profit_Status')
plt.title("Profitable vs Loss-Making Farms")
plt.show()

In [ ]:
# =====================================================================
# STEP 29: PERCENTAGE PROPORTIONAL PROFITABILITY PER SEASON
# =====================================================================
profit_status_season = pd.crosstab(df['Season'], df['Profit_Status'], normalize='index') * 100
display(profit_status_season)

profit_status_season.plot(kind='bar', figsize=(10,6))
plt.title("Profitability Percentage by Season")
plt.xlabel("Season")
plt.ylabel("Percentage (%)")
plt.xticks(rotation=0)
plt.show()

In [ ]:
# =====================================================================
# STEP 30: BIOTIC AND PEST THREAT LEVEL RISK SUMMARY
# =====================================================================
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x='Season', y='Disease_Pest_Risk_pct')
plt.title("Disease and Pest Risk Across Seasons")
plt.xlabel("Season")
plt.ylabel("Risk (%)")
plt.show()

risk_by_season = df.groupby('Season')['Disease_Pest_Risk_pct'].mean().sort_values(ascending=False)
display(risk_by_season)

In [ ]:
# =====================================================================
# STEP 31: WATER PRODUCTIVITY EFFICIENCY COMPARISONS
# =====================================================================
water_efficiency = df.groupby('Season')['Water_Efficiency_t_per_1000m3'].mean().sort_values(ascending=False)
display(water_efficiency)

plt.figure(figsize=(8,5))
sns.barplot(x=water_efficiency.index, y=water_efficiency.values)
plt.title("Water Efficiency Across Seasons")
plt.xlabel("Season")
plt.ylabel("Tonnes per 1000 m³")
plt.show()

In [ ]:
# =====================================================================
# STEP 32: LINEAR RELATIONSHIPS CORRELATION COEFF MATRIX
# =====================================================================
numerical_columns = [
    'Rainfall_mm', 'Avg_Temperature_C', 'Humidity_pct', 'Sunlight_Hours_Day',
    'Soil_pH', 'Soil_Moisture_pct', 'Nitrogen_kg_ha', 'Phosphorus_kg_ha', 'Potassium_kg_ha',
    'Fertilizer_kg_ha', 'Pesticide_Litre_ha', 'Seed_Quality_Score', 'Yield_Tonnes_Ha',
    'Production_Tonnes', 'Market_Price_INR_Tonne', 'Total_Cost_INR', 'Revenue_INR',
    'Profit_INR', 'Water_Used_m3', 'Water_Efficiency_t_per_1000m3', 'Disease_Pest_Risk_pct'
]

correlation = df[numerical_columns].corr()
plt.figure(figsize=(16,12))
sns.heatmap(correlation, cmap='coolwarm', center=0, annot=False)
plt.title("Correlation Matrix of Agricultural Variables")
plt.show()

In [ ]:
# =====================================================================
# STEP 33: CRITICAL COEFFICIENT DRIVERS INFLUENCING YIELD
# =====================================================================
yield_correlation = correlation['Yield_Tonnes_Ha'].sort_values(ascending=False)
print(yield_correlation)

In [ ]:
# =====================================================================
# STEP 34: CRITICAL COEFFICIENT DRIVERS INFLUENCING PROFITABILITY
# =====================================================================
profit_correlation = correlation['Profit_INR'].sort_values(ascending=False)
print(profit_correlation)

In [ ]:
# =====================================================================
# STEP 35: SCATTER EXPLORATION - RAINFALL IMPACT ON YIELD
# =====================================================================
plt.figure(figsize=(8,5))
sns.scatterplot(data=df, x='Rainfall_mm', y='Yield_Tonnes_Ha', hue='Season')
plt.title("Rainfall vs Crop Yield")
plt.xlabel("Rainfall (mm)")

plt.ylabel("Yield (Tonnes/Ha)")
plt.show()

In [ ]:
# =====================================================================
# STEP 36: SCATTER EXPLORATION - TEMPERATURE IMPACT ON YIELD
# =====================================================================
plt.figure(figsize=(8,5))
sns.scatterplot(data=df, x='Avg_Temperature_C', y='Yield_Tonnes_Ha', hue='Season')
plt.title("Temperature vs Crop Yield")
plt.xlabel("Average Temperature (°C)")
plt.ylabel("Yield (Tonnes/Ha)")
plt.show()

In [ ]:
# =====================================================================
# STEP 37: SCATTER EXPLORATION - CHEMICAL FERTILIZATION VS YIELD
# =====================================================================
plt.figure(figsize=(8,5))
sns.scatterplot(data=df, x='Fertilizer_kg_ha', y='Yield_Tonnes_Ha', hue='Season')
plt.title("Fertilizer Usage vs Yield")
plt.xlabel("Fertilizer (kg/ha)")
plt.ylabel("Yield (Tonnes/Ha)")
plt.show()

In [ ]:
# =====================================================================
# STEP 38: SCATTER EXPLORATION - BIOLOGICAL GENETIC SEED SCORE VS YIELD
# =====================================================================
plt.figure(figsize=(8,5))
sns.scatterplot(data=df, x='Seed_Quality_Score', y='Yield_Tonnes_Ha', hue='Season')
plt.title("Seed Quality vs Yield")
plt.xlabel("Seed Quality Score")
plt.ylabel("Yield (Tonnes/Ha)")
plt.show()

In [ ]:
# =====================================================================
# STEP 39: SCATTER EXPLORATION - VOLUME OF WATER USED VS YIELD
# =====================================================================
plt.figure(figsize=(8,5))
sns.scatterplot(data=df, x='Water_Used_m3', y='Yield_Tonnes_Ha', hue='Season')
plt.title("Water Usage vs Yield")
plt.xlabel("Water Used (m³)")
plt.ylabel("Yield (Tonnes/Ha)")
plt.show()

In [ ]:
# =====================================================================
# STEP 40: PARAMETRIC ANOVA TEST - DIFFERENCE IN YIELD BY SEASONS
# =====================================================================
season_groups = [group['Yield_Tonnes_Ha'].values for name, group in df.groupby('Season')]
anova_result = stats.f_oneway(*season_groups)
print("ANOVA F-statistic:", anova_result.statistic)
print("ANOVA p-value:", anova_result.pvalue)

alpha = 0.05
if anova_result.pvalue < alpha:
    print("Result: Yield differs significantly across seasons.")
else:
    print("Result: No statistically significant difference in yield across seasons.")

In [ ]:
# =====================================================================
# STEP 41: PARAMETRIC ANOVA TEST - DIFFERENCE IN PROFITABILITY BY SEASONS
# =====================================================================
profit_groups = [group['Profit_INR'].values for name, group in df.groupby('Season')]
anova_profit = stats.f_oneway(*profit_groups)
print("F-statistic:", anova_profit.statistic)
print("p-value:", anova_profit.pvalue)

if anova_profit.pvalue < 0.05:
    print("Profit differs significantly across seasons.")
else:
    print("No statistically significant difference in profit across seasons.")

In [ ]:
# =====================================================================
# STEP 42: EXECUTIVE MANAGEMENT SUMMARY DASHBOARD TABLE
# =====================================================================
dashboard = df.groupby('Season').agg(
    Farms=('Farm_ID', 'count'),
    Avg_Yield=('Yield_Tonnes_Ha', 'mean'),
    Avg_Production=('Production_Tonnes', 'mean'),
    Avg_Revenue=('Revenue_INR', 'mean'),
    Avg_Cost=('Total_Cost_INR', 'mean'),
    Avg_Profit=('Profit_INR', 'mean'),
    Avg_Water_Used=('Water_Used_m3', 'mean'),
    Avg_Water_Efficiency=('Water_Efficiency_t_per_1000m3', 'mean'),
    Avg_Pest_Risk=('Disease_Pest_Risk_pct', 'mean')
).round(2)
display(dashboard)

In [ ]:
# =====================================================================
# STEP 43: BENCHMARKING - TOP 10 STRATEGICALLY PROFITABLE FARMS
# =====================================================================
top_profitable_farms = df[
    ['Farm_ID', 'State', 'District', 'Crop', 'Season', 'Yield_Tonnes_Ha', 'Revenue_INR', 'Total_Cost_INR', 'Profit_INR']
].sort_values(by='Profit_INR', ascending=False).head(10)
display(top_profitable_farms)

In [ ]:
# =====================================================================
# STEP 44: BENCHMARKING - TOP 10 HIGH PERFORMING MAXIMUM YIELD FARMS
# =====================================================================
top_yield_farms = df[
    ['Farm_ID', 'State', 'District', 'Crop', 'Season', 'Yield_Tonnes_Ha', 'Production_Tonnes', 'Profit_INR']
].sort_values(by='Yield_Tonnes_Ha', ascending=False).head(10)
display(top_yield_farms)

In [ ]:
# =====================================================================
# STEP 45: CRITICAL DIAGNOSTICS - BOTTOM 10 LEAST PROFITABLE FARMS
# =====================================================================
lowest_profit_farms = df[
    ['Farm_ID', 'State', 'District', 'Crop', 'Season', 'Yield_Tonnes_Ha', 'Total_Cost_INR', 'Revenue_INR', 'Profit_INR']
].sort_values(by='Profit_INR').head(10)
display(lowest_profit_farms)

In [ ]:
# =====================================================================
# STEP 46: ML PREPARATION - CATEGORICAL FEATURE ENCODING
# =====================================================================
ml_df = df.copy()
categorical_features = ['State', 'District', 'Crop', 'Season', 'Irrigation_Method']
for col in categorical_features:
    encoder = LabelEncoder()
    ml_df[col] = encoder.fit_transform(ml_df[col].astype(str))

In [ ]:
# =====================================================================
# STEP 47: ML PREPARATION - DEFINE PREDICTIVE ATTRIBUTE FEATURES
# =====================================================================
features = [
    'Farm_Area_Hectares', 'Rainfall_mm', 'Avg_Temperature_C', 'Humidity_pct', 'Sunlight_Hours_Day',
    'Soil_pH', 'Soil_Moisture_pct', 'Nitrogen_kg_ha', 'Phosphorus_kg_ha', 'Potassium_kg_ha',
    'Fertilizer_kg_ha', 'Pesticide_Litre_ha', 'Seed_Quality_Score', 'Market_Price_INR_Tonne',
    'Water_Used_m3', 'Disease_Pest_Risk_pct', 'State', 'District', 'Crop', 'Season', 'Irrigation_Method'
]
X = ml_df[features]
y = ml_df['Yield_Tonnes_Ha']
print("Features shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
# =====================================================================
# STEP 48: ML PREPARATION - STRATIFIED TRAIN-TEST SPLITTING
# =====================================================================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

In [ ]:
# =====================================================================
# STEP 49: MODEL DEVELOPMENT - RANDOM FOREST REGRESSION ENSEMBLE
# =====================================================================
model = RandomForestRegressor(n_estimators=200, random_state=42)
model.fit(X_train, y_train)
print("Model trained successfully!")

In [ ]:
# =====================================================================
# STEP 50: MODEL GENERALIZATION EVALUATION STATS (MAE, RMSE, R2)
# =====================================================================
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Mean Absolute Error:", mae)
print("Root Mean Squared Error:", rmse)
print("R² Score:", r2)

In [ ]:
# =====================================================================
# STEP 51: ENSEMBLE FEATURE IMPORTANCE SPECTRUM ANALYSIS
# =====================================================================
feature_importance = pd.DataFrame({
    'Feature': features,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

display(feature_importance)

plt.figure(figsize=(10,8))
sns.barplot(data=feature_importance.head(15), x='Importance', y='Feature')
plt.title("Top Factors Influencing Crop Yield")
plt.show()

In [ ]:
# =====================================================================
# STEP 52: EVALUATION PLOT - GROUND TRUTH VS MODEL PREDICTIONS
# =====================================================================
plt.figure(figsize=(8,6))
plt.scatter(y_test, y_pred, alpha=0.6)
plt.xlabel("Actual Yield")
plt.ylabel("Predicted Yield")
plt.title("Actual vs Predicted Crop Yield")
plt.show()

In [ ]:
# =====================================================================
# STEP 53: KNOWLEDGE RETRIEVAL - EXECUTIVE KEY FINDINGS AUTOMATION
# =====================================================================
best_yield_season = seasonal_summary['Yield_Tonnes_Ha'].idxmax()
best_profit_season = seasonal_summary['Profit_INR'].idxmax()
best_revenue_season = seasonal_summary['Revenue_INR'].idxmax()
best_water_season = seasonal_summary['Water_Efficiency_t_per_1000m3'].idxmax()
highest_risk_season = seasonal_summary['Disease_Pest_Risk_pct'].idxmax()
highest_cost_season = seasonal_summary['Total_Cost_INR'].idxmax()

print("========== KEY FINDINGS ==========")
print(f"1. Highest average yield was observed in {best_yield_season}.")
print(f"2. Highest average profit was observed in {best_profit_season}.")
print(f"3. Highest average revenue was observed in {best_revenue_season}.")
print(f"4. Best average water efficiency was observed in {best_water_season}.")
print(f"5. Highest average disease/pest risk was observed in {highest_risk_season}.")
print(f"6. Highest average agricultural cost was observed in {highest_cost_season}.")

In [ ]:
# =====================================================================
# STEP 54: SEASONAL DEVIATION COMPARATIVE METRICS MATRIX
# =====================================================================
final_comparison = df.groupby('Season').agg({
    'Rainfall_mm': 'mean',
    'Avg_Temperature_C': 'mean',
    'Humidity_pct': 'mean',
    'Yield_Tonnes_Ha': 'mean',
    'Production_Tonnes': 'mean',
    'Total_Cost_INR': 'mean',
    'Revenue_INR': 'mean',
    'Profit_INR': 'mean',
    'Water_Used_m3': 'mean',
    'Water_Efficiency_t_per_1000m3': 'mean',
    'Disease_Pest_Risk_pct': 'mean'
}).round(2)
display(final_comparison)

In [ ]:
# =====================================================================
# STEP 55: DATA EXPORT - EXCEL FORMAT REPOSITORY SYSTEM
# =====================================================================
import openpyxl
with pd.ExcelWriter('seasonal_agriculture_analysis_results.xlsx', engine='openpyxl') as writer:
    seasonal_summary.to_excel(writer, sheet_name='Seasonal Summary')
    crop_season_yield.to_excel(writer, sheet_name='Crop Season Yield')
    season_crop_profit.to_excel(writer, sheet_name='Crop Season Profit')
    dashboard.to_excel(writer, sheet_name='Dashboard')
    feature_importance.to_excel(writer, sheet_name='Feature Importance', index=False)
print("Excel report created successfully!")

In [ ]:
# =====================================================================
# STEP 56: OVERALL STRATEGIC PROJECT CONCLUSION
# =====================================================================
print("""
=============================================
              PROJECT CONCLUSION
=============================================
This project analyzed agricultural performance
across different seasons using the provided
agricultural dataset.

The analysis examined:
1. Seasonal agricultural performance
2. Crop yield and production
3. Revenue, cost and profitability
4. Environmental conditions
5. Soil characteristics
6. Fertilizer and pesticide usage
7. Irrigation methods
8. Water consumption and efficiency
9. Disease and pest risk
10. Relationships between agricultural variables
11. Statistical differences between seasons
12. Factors associated with crop yield

The analysis provides evidence-based insights
into seasonal differences and can support better
agricultural planning and resource management.
=============================================
""")